# 🌊 Flood Extent Mapping from Sentinel-1 SAR Imagery
## DeepLabV3+ · EfficientNet-B5 · Weak Label Pre-training · TTA

**Dataset:** Sen1Floods11 (Official Benchmark)  
**Final Test IoU: 0.5907 | Recall: 0.7403** (90 official test chips)

---

### Pipeline Overview
```
Phase 1: Pre-train on 4,385 weakly-labeled images  →  learn global water patterns
Phase 2: Fine-tune on 446 hand-labeled images      →  precise flood detection  
Phase 3: Evaluate on official test split with TTA  →  honest benchmark results
```

### Why SAR Instead of Optical?
Floods happen during storms. Storms = clouds. Clouds block optical cameras completely.  
Sentinel-1 SAR radar works **through clouds, through rain, at night** — exactly when you need it.

| Method | IoU | Notes |
|--------|-----|-------|
| Prithvi-CAFE (WACV 2026) | 83.41 | NASA/IBM foundation model |
| Baseline U-Net (literature) | 70.57 | Standard architecture |
| Simple threshold | ~35.00 | No deep learning |
| **This notebook** | **59.07** | Single Colab T4 GPU |



## ⚙️ 1. Setup and Installation

In [ ]:
# Install required libraries
!pip install -q segmentation-models-pytorch albumentations rasterio \
             rioxarray torchmetrics scikit-learn

import torch
import torch.nn as nn
import numpy as np
import rasterio
import glob
import os
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2
import segmentation_models_pytorch as smp
from torchmetrics.classification import BinaryJaccardIndex, BinaryRecall

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ Running on: {DEVICE}")
print(f"PyTorch version: {torch.__version__}")


## 📥 2. Download Sen1Floods11 Dataset

Sen1Floods11 contains globally distributed flood events across 6 continents.  
- **Hand-labeled:** 446 chips with human expert annotations  
- **Weakly-labeled:** 4,385 chips with Otsu algorithm labels  
- **Test split:** 90 chips (official benchmark — never used during training)


In [ ]:
os.makedirs('/content/sen1floods11', exist_ok=True)
os.makedirs('/content/sen1floods11/WeakData', exist_ok=True)

# Download Hand-Labeled data (446 chips)
print("📥 Downloading hand-labeled SAR images...")
!gsutil -m cp -n -r gs://sen1floods11/v1.1/data/flood_events/HandLabeled/S1Hand /content/sen1floods11/
!gsutil -m cp -n -r gs://sen1floods11/v1.1/data/flood_events/HandLabeled/LabelHand /content/sen1floods11/

# Download Weakly-Labeled data (4,385 chips)
print("\n📥 Downloading weakly-labeled SAR images (takes ~5 minutes)...")
!gsutil -m cp -n -r gs://sen1floods11/v1.1/data/flood_events/WeaklyLabeled/S1Weak /content/sen1floods11/WeakData/
!gsutil -m cp -n -r gs://sen1floods11/v1.1/data/flood_events/WeaklyLabeled/S1OtsuLabelWeak /content/sen1floods11/WeakData/

# Download official test split CSV
print("\n📥 Downloading official test split...")
!gsutil cp gs://sen1floods11/v1.1/splits/flood_handlabeled/flood_test_data.csv .

print("\n✅ All downloads complete!")


## 🗂️ 3. Data Pairing and Splitting

We pair SAR images with their labels using filename matching.  

**Important:** Validation uses ONLY human-labeled data — never weak labels.  
Weak labels are used only for training (Phase 1 pre-training).


In [ ]:
def get_base_id(filepath):
    """Extract region_chipID from filename e.g. Bolivia_12345 from Bolivia_12345_S1Weak.tif"""
    basename = os.path.basename(filepath)
    parts = basename.split('_')
    return f"{parts[0]}_{parts[1]}" if len(parts) >= 2 else basename

# ── Find all files ──────────────────────────────────────────────
all_hand_s1  = sorted(glob.glob('/content/sen1floods11/S1Hand/*_S1Hand.tif'))
all_hand_lbl = sorted(glob.glob('/content/sen1floods11/LabelHand/*_LabelHand.tif'))
all_weak_s1  = sorted(glob.glob('/content/sen1floods11/WeakData/S1Weak/*_S1Weak.tif'))
all_weak_lbl = sorted(glob.glob('/content/sen1floods11/WeakData/S1OtsuLabelWeak/*_S1OtsuLabelWeak.tif'))

# ── Safe pairing by filename ID ─────────────────────────────────
hand_lbl_dict = {os.path.basename(f).replace('_LabelHand.tif', ''): f for f in all_hand_lbl}
weak_lbl_dict = {get_base_id(f): f for f in all_weak_lbl}

clean_hand_imgs, clean_hand_lbls = [], []
for img in all_hand_s1:
    base = os.path.basename(img).replace('_S1Hand.tif', '')
    if base in hand_lbl_dict:
        clean_hand_imgs.append(img)
        clean_hand_lbls.append(hand_lbl_dict[base])

clean_weak_imgs, clean_weak_lbls = [], []
for img in all_weak_s1:
    base = get_base_id(img)
    if base in weak_lbl_dict:
        clean_weak_imgs.append(img)
        clean_weak_lbls.append(weak_lbl_dict[base])

# ── Split hand-labeled: 80% train, 20% val ─────────────────────
hand_train_imgs, val_imgs, hand_train_lbls, val_lbls = train_test_split(
    clean_hand_imgs, clean_hand_lbls, test_size=0.2, random_state=42
)

# ── Phase 1 training: weak + hand combined ─────────────────────
phase1_train_imgs = clean_weak_imgs + hand_train_imgs
phase1_train_lbls = clean_weak_lbls + hand_train_lbls

print(f"✅ Hand-labeled pairs:  {len(clean_hand_imgs)}")
print(f"✅ Weakly-labeled pairs: {len(clean_weak_imgs)}")
print(f"\n📊 DATA SPLIT:")
print(f"  Phase 1 training: {len(phase1_train_imgs)} images (weak + hand)")
print(f"  Phase 2 training: {len(hand_train_imgs)} images (hand only)")
print(f"  Validation:       {len(val_imgs)} images (hand only — strict)")


## 🛰️ 4. Dataset and Augmentations

### SAR Preprocessing
```
Raw SAR (dB)  →  clip[-30, 0]  →  normalize center -15dB  →  handle NaN
```
- Water is very dark: ~-25 to -30 dB (smooth surface, signal bounces away)
- Vegetation is moderate: ~-12 to -8 dB
- Urban is bright: ~-5 to 0 dB (corner reflectors bounce signal back)

### Augmentations
Satellite images have no natural "up" orientation — rotations and flips are always valid.


In [ ]:
class Sen1FloodsDataset(Dataset):
    """
    PyTorch Dataset for Sen1Floods11 SAR flood detection.
    
    Handles both hand-labeled and weakly-labeled data with
    identical preprocessing — ensures consistent model input.
    """
    def __init__(self, images, labels, transform=None):
        self.images = images
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        # Load SAR image (VV + VH bands)
        with rasterio.open(self.images[idx]) as src:
            sar = src.read()
            sar = np.nan_to_num(sar, nan=0.0)

        # SAR normalization: clip to physical range, center at -15dB
        sar = np.clip(sar, -30, 0)
        sar = (sar - (-15.0)) / 10.0
        sar = np.transpose(sar, (1, 2, 0))  # (C,H,W) → (H,W,C) for albumentations

        # Load flood mask (-1=nodata, 0=dry, 1=flooded)
        with rasterio.open(self.labels[idx]) as src:
            mask = src.read(1)
            mask = np.where(mask == 1, 1.0, 0.0).astype(np.float32)

        if self.transform:
            aug = self.transform(image=sar, mask=mask)
            sar, mask = aug['image'], aug['mask']

        return sar, mask.unsqueeze(0)


# Training augmentations — random crops + flips + rotations
train_transform = A.Compose([
    A.RandomCrop(width=256, height=256),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    ToTensorV2()
])

# Validation transform — full resolution
val_transform = A.Compose([
    A.Resize(512, 512),
    ToTensorV2()
])

# Quick sanity check
test_ds = Sen1FloodsDataset(clean_hand_imgs[:5], clean_hand_lbls[:5], val_transform)
img, msk = test_ds[0]
print(f"✅ Image shape: {img.shape}  (channels=2: VV+VH)")
print(f"✅ Mask shape:  {msk.shape}  (binary: 0=dry, 1=flood)")
print(f"✅ Image range: {img.min():.2f} to {img.max():.2f}  (normalized)")
print(f"✅ Mask values: {msk.unique()}  (binary confirmed)")


## 🏗️ 5. Model Architecture — DeepLabV3+ with EfficientNet-B5

### Why DeepLabV3+ over standard U-Net?

DeepLabV3+ uses **Atrous Spatial Pyramid Pooling (ASPP)** — looks at the image at 4 different scales simultaneously.

This is critical for SAR flood mapping:
- **Local scale:** is this pixel dark? (water signature)
- **Neighborhood scale:** are surrounding pixels also dark?
- **Regional scale:** is this near a river or coast?
- **Scene scale:** is this a flood plain? what's the terrain?

Combining all 4 scales helps distinguish dark water from dark shadows, roads, and urban areas.

### Why EfficientNet-B5 with noisy-student weights?
Pretrained on 300M images using semi-supervised learning — excellent texture recognition.


In [ ]:
model = smp.DeepLabV3Plus(
    encoder_name="timm-efficientnet-b5",
    encoder_weights="noisy-student",   # pretrained on 300M images
    in_channels=2,                     # VV + VH SAR channels
    classes=1                          # binary: flood or not
).to(DEVICE)

total_params = sum(p.numel() for p in model.parameters())
print(f"✅ Model: DeepLabV3+ with EfficientNet-B5 encoder")
print(f"✅ Total parameters: {total_params:,}")
print(f"✅ Input: (batch, 2, 512, 512)  — VV + VH")
print(f"✅ Output: (batch, 1, 512, 512) — flood probability map")

# Loss function: Tversky + BCE
# Tversky(alpha=0.3, beta=0.7) penalizes missed floods 2.3x more than false alarms
# This reflects real-world cost: missing a flood is more dangerous than a false alarm
tversky_loss = smp.losses.TverskyLoss(
    smp.losses.BINARY_MODE, alpha=0.3, beta=0.7, from_logits=True
)
bce_loss = nn.BCEWithLogitsLoss()

def criterion(pred, target):
    return tversky_loss(pred, target) + (0.5 * bce_loss(pred, target))

print("\n✅ Loss: Tversky(α=0.3, β=0.7) + 0.5×BCE")
print("   → Penalizes missed floods 2.3× more than false alarms")


## 🚀 6. Phase 1 — Weak Label Pre-training

Train on 4,385 weakly-labeled images to learn global water appearance patterns.

**Strategy:** Start broad, learn general water/land discrimination globally.  
**Loss:** Tversky + BCE  
**Scheduler:** CosineAnnealingWarmRestarts for smooth learning rate cycling


In [ ]:
# Phase 1 DataLoaders
phase1_train_loader = DataLoader(
    Sen1FloodsDataset(phase1_train_imgs, phase1_train_lbls, train_transform),
    batch_size=4, shuffle=True, num_workers=2
)
val_loader = DataLoader(
    Sen1FloodsDataset(val_imgs, val_lbls, val_transform),
    batch_size=2, shuffle=False
)

optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-3)
scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
    optimizer, T_0=5, T_mult=2
)
iou_metric    = BinaryJaccardIndex().to(DEVICE)
recall_metric = BinaryRecall().to(DEVICE)

best_iou  = 0.0
# ── KEY FIX: only 5 epochs ──────────────────────────────────────
# Weak labels are noisy (Otsu algorithm, not human expert)
# Model peaks early — more epochs = overfitting on noise
# Phase 2 will correct with clean human labels
EPOCHS_P1 = 5

print("🚀 Phase 1: Pre-training on weak + hand labels...")
print(f"{'Epoch':>6} | {'Train Loss':>10} | {'Val IoU':>8} | {'Recall':>8} | {'Status':>15}")
print("-" * 60)

for epoch in range(EPOCHS_P1):
    # ── Training ────────────────────────────────────────────────
    model.train()
    train_loss = 0
    for images, masks in phase1_train_loader:
        images, masks = images.to(DEVICE), masks.to(DEVICE)
        optimizer.zero_grad()
        logits = model(images)
        loss = criterion(logits, masks)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        train_loss += loss.item()

    # ── Validation with TTA ─────────────────────────────────────
    model.eval()
    iou_metric.reset()
    recall_metric.reset()

    with torch.no_grad():
        for images, masks in val_loader:
            images, masks = images.to(DEVICE), masks.to(DEVICE)
            p_normal = torch.sigmoid(model(images))
            p_hflip  = torch.flip(torch.sigmoid(model(torch.flip(images, [3]))), [3])
            p_vflip  = torch.flip(torch.sigmoid(model(torch.flip(images, [2]))), [2])
            final_preds = (p_normal + p_hflip + p_vflip) / 3.0
            iou_metric.update(final_preds, masks.int())
            recall_metric.update(final_preds, masks.int())

    val_iou    = iou_metric.compute().item()
    val_recall = recall_metric.compute().item()
    avg_loss   = train_loss / len(phase1_train_loader)
    scheduler.step()

    status = ""
    if val_iou > best_iou:
        best_iou = val_iou
        torch.save(model.state_dict(), "phase1_best.pth")
        status = "✅ saved"

    print(f"{epoch+1:>6} | {avg_loss:>10.4f} | {val_iou:>8.4f} | "
          f"{val_recall:>8.4f} | {status:>15}")

print(f"\n✅ Phase 1 complete. Best Val IoU: {best_iou:.4f}")
print(f"   Checkpoint saved: phase1_best.pth")
print(f"   → Now running Phase 2 fine-tuning on human labels...")


## 🎯 7. Phase 2 — Fine-tuning on Human Labels

Load the Phase 1 checkpoint and fine-tune on **357 expert hand-labeled images**.

### What changed vs Phase 1

| | Phase 1 | Phase 2 |
|--|---------|---------|
| Data | 4,800 weak + hand labels | 357 human labels only |
| Purpose | Learn SAR water appearance | Precise flood boundaries |
| LR | 3e-4 | 1e-4 → decays with scheduler |
| Epochs | 5 (stop before noisy overfit) | 30 (enough time to converge) |
| Scheduler | CosineAnnealing | ReduceLROnPlateau (patience=5) |

### Why ReduceLROnPlateau for Phase 2?
Phase 2 has far less data (357 vs 4,800 images).  
The model needs the LR to automatically drop when it stops improving —  
otherwise it overshoots the optimal weights on this small dataset.


In [ ]:
# ── Load Phase 1 best weights ──────────────────────────────────
model.load_state_dict(torch.load("phase1_best.pth", weights_only=True))
print("✅ Phase 1 weights loaded")
print(f"   Starting Val IoU from Phase 1: {best_iou:.4f}")

# ── Phase 2 DataLoaders — hand-labeled only ─────────────────────
finetune_loader = DataLoader(
    Sen1FloodsDataset(hand_train_imgs, hand_train_lbls, train_transform),
    batch_size=4, shuffle=True
)
val_loader_ft = DataLoader(
    Sen1FloodsDataset(val_imgs, val_lbls, val_transform),
    batch_size=2, shuffle=False
)

# ── Optimizer: higher LR than before, scheduler will reduce it ──
# lr=1e-4 instead of 1e-5 — gives more room to improve
# Scheduler reduces LR when Val IoU plateaus (patience=5 epochs)
optimizer_ft  = torch.optim.AdamW(model.parameters(),
                                   lr=1e-4, weight_decay=1e-4)
scheduler_ft  = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer_ft, mode='max', factor=0.5, patience=5
)

iou_metric_ft    = BinaryJaccardIndex().to(DEVICE)
recall_metric_ft = BinaryRecall().to(DEVICE)

# ── KEY FIX: 30 epochs (was 15) ────────────────────────────────
# 357 human-labeled images need more epochs to converge fully
# Scheduler + early stopping protect against overfitting
EPOCHS_P2     = 30
best_iou_p2   = 0.0   # track Phase 2 best separately
no_improve    = 0     # early stopping counter
PATIENCE      = 10    # stop if no improvement for 10 epochs

print(f"\n🎯 Phase 2: Fine-tuning on {len(hand_train_imgs)} human expert labels...")
print(f"{'Epoch':>6} | {'Train Loss':>10} | {'Val IoU':>8} | {'Recall':>8} | {'LR':>8} | {'Status':>14}")
print("-" * 75)

for epoch in range(EPOCHS_P2):
    # ── Training ────────────────────────────────────────────────
    model.train()
    train_loss = 0
    for imgs, msks in finetune_loader:
        imgs, msks = imgs.to(DEVICE), msks.to(DEVICE)
        optimizer_ft.zero_grad()
        loss = criterion(model(imgs), msks)
        loss.backward()
        # ── KEY FIX: gradient clipping (was missing in Phase 2) ──
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer_ft.step()
        train_loss += loss.item()

    # ── Validation with TTA ─────────────────────────────────────
    model.eval()
    iou_metric_ft.reset()
    recall_metric_ft.reset()

    with torch.no_grad():
        for imgs, msks in val_loader_ft:
            imgs, msks = imgs.to(DEVICE), msks.to(DEVICE)
            p_normal = torch.sigmoid(model(imgs))
            p_hflip  = torch.flip(torch.sigmoid(model(torch.flip(imgs, [3]))), [3])
            p_vflip  = torch.flip(torch.sigmoid(model(torch.flip(imgs, [2]))), [2])
            preds = (p_normal + p_hflip + p_vflip) / 3.0
            iou_metric_ft.update(preds, msks.int())
            recall_metric_ft.update(preds, msks.int())

    current_iou    = iou_metric_ft.compute().item()
    current_recall = recall_metric_ft.compute().item()
    avg_loss       = train_loss / len(finetune_loader)
    current_lr     = optimizer_ft.param_groups[0]['lr']

    # ── Scheduler step on Val IoU ────────────────────────────────
    scheduler_ft.step(current_iou)

    # ── Save best + early stopping ───────────────────────────────
    status = ""
    if current_iou > best_iou_p2:
        best_iou_p2 = current_iou
        best_iou    = current_iou   # update global best
        no_improve  = 0
        torch.save(model.state_dict(), "SOTA_Flood_Model_Final.pth")
        status = "⭐ best saved"
    else:
        no_improve += 1
        if no_improve >= PATIENCE:
            print(f"\n⏹️  Early stopping at epoch {epoch+1} "
                  f"(no improvement for {PATIENCE} epochs)")
            break

    print(f"{epoch+1:>6} | {avg_loss:>10.4f} | {current_iou:>8.4f} | "
          f"{current_recall:>8.4f} | {current_lr:>8.2e} | {status:>14}")

print(f"\n✅ Phase 2 complete.")
print(f"   Best Val IoU:  {best_iou_p2:.4f}")
print(f"   Checkpoint:    SOTA_Flood_Model_Final.pth")


## 📊 8. Official Test Set Evaluation

Evaluate on the 90 official test chips — **touched only once, at the very end.**

This is the honest number that goes in the README and CV.  
Test-Time Augmentation (3 orientations) is applied for best performance.


In [ ]:
# Load official test split
test_df = pd.read_csv('flood_test_data.csv', header=None)
test_imgs = [f"/content/sen1floods11/S1Hand/{f}" for f in test_df[0].tolist()]
test_lbls = [f"/content/sen1floods11/LabelHand/{f}" for f in test_df[1].tolist()]

test_loader = DataLoader(
    Sen1FloodsDataset(test_imgs, test_lbls, val_transform),
    batch_size=1, shuffle=False
)

# Load best weights

model.load_state_dict(torch.load("SOTA_Flood_Model_Final.pth"))
model.eval()

test_iou    = BinaryJaccardIndex().to(DEVICE)
test_recall = BinaryRecall().to(DEVICE)

print(f"🧐 Evaluating on {len(test_loader)} official test chips...")

with torch.no_grad():
    for images, masks in test_loader:
        images, masks = images.to(DEVICE), masks.to(DEVICE)

        # TTA: 3-orientation ensemble
        p1 = torch.sigmoid(model(images))
        p2 = torch.flip(torch.sigmoid(model(torch.flip(images, [3]))), [3])
        p3 = torch.flip(torch.sigmoid(model(torch.flip(images, [2]))), [2])
        final_preds = (p1 + p2 + p3) / 3.0

        test_iou.update(final_preds, masks.int())
        test_recall.update(final_preds, masks.int())

print("=" * 35)
print(f"🔥 FINAL TEST IoU:    {test_iou.compute().item():.4f}")
print(f"🔥 FINAL TEST RECALL: {test_recall.compute().item():.4f}")
print("=" * 35)
print("\nInterpretation:")
print(f"  The model finds {test_recall.compute().item()*100:.1f}% of all flooded areas")
print(f"  IoU measures overlap between predicted and true flood masks")


## 🖼️ 9. Visualize Predictions

Four-panel visualization:
1. SAR VV input (radar backscatter — dark = smooth surfaces like water)
2. SAR VH input (cross-polarization — highlights vegetation)
3. Human ground truth mask
4. Model prediction


In [ ]:
def visualize_predictions(model, dataset, num_samples=3, threshold=0.5):
    model.eval()
    fig, axes = plt.subplots(num_samples, 4, figsize=(18, 4.5*num_samples))

    with torch.no_grad():
        for i in range(num_samples):
            idx = np.random.randint(0, len(dataset))
            image, mask = dataset[idx]

            input_tensor = image.unsqueeze(0).to(DEVICE)
            logits = model(input_tensor)
            prob_map  = torch.sigmoid(logits).squeeze().cpu().numpy()
            pred_mask = (prob_map > threshold).astype(float)

            vv = image[0].numpy()
            vh = image[1].numpy()
            gt = mask.squeeze().numpy()
            
            iou_val = (
                np.logical_and(pred_mask, gt).sum() /
                (np.logical_or(pred_mask, gt).sum() + 1e-6)
            )

            axes[i, 0].imshow(vv, cmap='gray')
            axes[i, 0].set_title('SAR VV (Input)', fontweight='bold')
            axes[i, 0].axis('off')

            axes[i, 1].imshow(vh, cmap='gray')
            axes[i, 1].set_title('SAR VH (Input)', fontweight='bold')
            axes[i, 1].axis('off')

            axes[i, 2].imshow(gt, cmap='Blues', vmin=0, vmax=1)
            axes[i, 2].set_title('Ground Truth', fontweight='bold')
            axes[i, 2].axis('off')

            axes[i, 3].imshow(pred_mask, cmap='Reds', vmin=0, vmax=1)
            axes[i, 3].set_title(f'Prediction  |  IoU: {iou_val:.3f}', fontweight='bold')
            axes[i, 3].axis('off')

    plt.suptitle('Flood Detection — Model Predictions vs Ground Truth',
                 fontsize=14, fontweight='bold', y=1.01)
    plt.tight_layout()
    plt.savefig('predictions.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("✅ Saved: predictions.png")

val_dataset = Sen1FloodsDataset(val_imgs, val_lbls, val_transform)
visualize_predictions(model, val_dataset, num_samples=4)


## 🛰️ 10. Verification with Sentinel-2 Optical Imagery

Cross-validate SAR predictions against coincident Sentinel-2 optical imagery.

This 4-panel plot shows:
1. **SAR input** — what the model sees (radar backscatter)
2. **Optical verification** — what the area actually looks like (false color)  
3. **MNDWI water index** — water detected from optical (ground truth)
4. **Model prediction** — what our SAR model predicted

This verifies the model works physically, not just statistically.


In [ ]:

import os

s2_dir     = '/content/sen1floods11/WeakData/S2Weak'
s2_lbl_dir = '/content/sen1floods11/WeakData/S2IndexLabelWeak'

if not os.path.exists(s2_dir) or len(os.listdir(s2_dir)) == 0:
    print("📥 Downloading Sentinel-2 verification imagery...")
    !gsutil -m cp -n -r gs://sen1floods11/v1.1/data/flood_events/WeaklyLabeled/S2Weak /content/sen1floods11/WeakData/
    print("✅ S2 imagery downloaded")
else:
    print(f"✅ S2 imagery already present ({len(os.listdir(s2_dir))} files)")

if not os.path.exists(s2_lbl_dir) or len(os.listdir(s2_lbl_dir)) == 0:
    print("📥 Downloading S2 water index labels...")
    !gsutil -m cp -n -r gs://sen1floods11/v1.1/data/flood_events/WeaklyLabeled/S2IndexLabelWeak /content/sen1floods11/WeakData/
    print("✅ S2 labels downloaded")
else:
    print(f"✅ S2 labels already present ({len(os.listdir(s2_lbl_dir))} files)")


def verify_with_s2(model, chip_id):
    """
    Four-panel verification: SAR input | S2 optical | MNDWI label | Model prediction
    chip_id: the numeric ID portion of the filename e.g. '1009032'
    """
    # ── Find files safely ────────────────────────────────────────
    s1_files  = glob.glob(f'/content/sen1floods11/WeakData/S1Weak/*_{chip_id}_S1Weak.tif')
    s2_files  = glob.glob(f'/content/sen1floods11/WeakData/S2Weak/*_{chip_id}_S2Weak.tif')
    lbl_files = glob.glob(f'/content/sen1floods11/WeakData/S2IndexLabelWeak/*_{chip_id}_S2IndexLabelWeak.tif')

    if not s1_files:
        print(f"❌ S1 file not found for chip_id='{chip_id}'")
        print("   Try a different chip_id — run list_available_chips() to see options")
        return
    if not s2_files:
        print(f"❌ S2 optical file not found for chip_id='{chip_id}'")
        print("   S2 data may not cover this chip — try a different chip_id")
        return
    if not lbl_files:
        print(f"❌ S2 label file not found for chip_id='{chip_id}'")
        return

    s1_path, s2_path, s2_lbl_path = s1_files[0], s2_files[0], lbl_files[0]

    model.eval()

    # ── Load and preprocess SAR ──────────────────────────────────
    with rasterio.open(s1_path) as src:
        s1_img = src.read()
        s1_pre = np.nan_to_num(s1_img, nan=0.0)
        s1_pre = (np.clip(s1_pre, -30, 0) - (-15.0)) / 10.0

    # ── Predict with TTA ─────────────────────────────────────────
    with torch.no_grad():
        t = torch.from_numpy(s1_pre).float().unsqueeze(0).to(DEVICE)
        p1 = torch.sigmoid(model(t))
        p2 = torch.flip(torch.sigmoid(model(torch.flip(t, [3]))), [3])
        p3 = torch.flip(torch.sigmoid(model(torch.flip(t, [2]))), [2])
        pred = ((p1 + p2 + p3) / 3.0 > 0.5).cpu().numpy()[0][0]

    # ── Load S2 optical — false color (SWIR, NIR, Red) ──────────
    with rasterio.open(s2_path) as src:
        s2_img = src.read([11, 8, 4]).astype(float) / 10000.0
        s2_img = np.clip(s2_img * 2.5, 0, 1)
        s2_img = np.transpose(s2_img, (1, 2, 0))

    # ── Load MNDWI water label ───────────────────────────────────
    with rasterio.open(s2_lbl_path) as src:
        s2_label = src.read(1)

    # ── Plot ─────────────────────────────────────────────────────
    fig, axes = plt.subplots(1, 4, figsize=(20, 5))
    axes[0].imshow(s1_img[0], cmap='gray')
    axes[0].set_title('S1 SAR Input\n(VV backscatter)', fontweight='bold')
    axes[1].imshow(s2_img)
    axes[1].set_title('S2 Optical\n(SWIR/NIR/Red false color)', fontweight='bold')
    axes[2].imshow(s2_label, cmap='Blues')
    axes[2].set_title('MNDWI Water Label\n(from optical)', fontweight='bold')
    axes[3].imshow(pred, cmap='Blues')
    axes[3].set_title('Model Prediction\n(from SAR only)', fontweight='bold')
    for ax in axes:
        ax.axis('off')
    plt.suptitle(
        f'Cross-Validation: SAR Model vs Optical Reference  |  Chip: {chip_id}',
        fontsize=12, fontweight='bold'
    )
    plt.tight_layout()
    plt.savefig(f'verification_{chip_id}.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f"✅ Saved: verification_{chip_id}.png")


def list_available_chips(n=10):
    """Print chip IDs available in both S1 and S2 data."""
    import re
    s1_files = glob.glob('/content/sen1floods11/WeakData/S1Weak/*.tif')
    s2_files = glob.glob('/content/sen1floods11/WeakData/S2Weak/*.tif')

    # Extract IDs
    s1_ids = set()
    for f in s1_files:
        m = re.search(r'_(\d+)_S1Weak', os.path.basename(f))
        if m: s1_ids.add(m.group(1))

    s2_ids = set()
    for f in s2_files:
        m = re.search(r'_(\d+)_S2Weak', os.path.basename(f))
        if m: s2_ids.add(m.group(1))

    both = sorted(s1_ids & s2_ids)[:n]
    print(f"Chip IDs with both S1 and S2 data ({len(s1_ids & s2_ids)} total):")
    for cid in both:
        print(f"  verify_with_s2(model, '{cid}')")
    return both


# ── Run verification ─────────────────────────────────────────────
# First find a valid chip_id that exists in both S1 and S2
available = list_available_chips(n=5)
if available:
    verify_with_s2(model, available[0])


## 🔮 11. Production Inference — Any Sentinel-1 GeoTIFF

Run the trained model on any new Sentinel-1 image and export a GeoTIFF flood map.

The output GeoTIFF **preserves the coordinate system** of the input — it can be  
loaded directly in QGIS, ArcGIS, or any GIS software and placed correctly on Earth.


In [ ]:
def run_flood_inference(model, input_tif, output_tif, threshold=0.5):
    """
    Run flood detection on any Sentinel-1 GRD GeoTIFF.
    
    Args:
        model:      Trained DeepLabV3+ model
        input_tif:  Path to Sentinel-1 GeoTIFF (needs VV and VH bands)
        output_tif: Output path for flood mask GeoTIFF
        threshold:  Probability threshold (default 0.5, lower = more sensitive)
    
    Returns:
        Binary numpy array (1=flooded, 0=dry)
    """
    model.eval()

    with rasterio.open(input_tif) as src:
        image = src.read([1, 2])   # read VV (band 1) and VH (band 2)
        meta  = src.meta

    # Preprocess
    image = np.nan_to_num(image, nan=0.0)
    image = np.clip(image, -30, 0)
    image = (image - (-15.0)) / 10.0

    # Predict with TTA
    input_tensor = torch.from_numpy(image).float().unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        p1 = torch.sigmoid(model(input_tensor))
        p2 = torch.flip(torch.sigmoid(model(torch.flip(input_tensor, [3]))), [3])
        p3 = torch.flip(torch.sigmoid(model(torch.flip(input_tensor, [2]))), [2])
        avg_probs = (p1 + p2 + p3) / 3.0
        mask = (avg_probs > threshold).cpu().numpy().astype(np.uint8)[0][0]

    # Export GeoTIFF with original coordinate system preserved
    meta.update(dtype=rasterio.uint8, count=1, nodata=0)
    with rasterio.open(output_tif, 'w', **meta) as dst:
        dst.write(mask, 1)

    # Calculate flooded area
    # Sentinel-1 chips are always 10m × 10m — hardcoded because
    # rasterio res returns degrees (EPSG:4326), not metres
    flood_pixels = np.count_nonzero(mask)
    area_km2     = flood_pixels * 100 / 1_000_000  # 100 m² per pixel → km²

    print(f"✅ Flood map saved: {output_tif}")
    print(f"🌊 Flooded pixels:  {flood_pixels:,}")
    print(f"🌊 Flooded area:    {area_km2:.4f} km²")
    return mask

# Run on first test image as example
mask = run_flood_inference(
    model=model,
    input_tif=test_imgs[0],
    output_tif="flood_map_output.tif"
)


## 📐 12. Total Flooded Area Across Test Set

Aggregate flood area across all 90 test chips to estimate total impact.


In [ ]:
total_flood_pixels = 0
total_area_km2     = 0.0

# Sentinel-1 GRD chips in Sen1Floods11 are 10m × 10m pixels
# We hardcode this rather than reading res from rasterio,
# because rasterio returns degrees (EPSG:4326) not metres for these files
PIXEL_SIZE_M   = 10.0                        # metres
PIXEL_AREA_M2  = PIXEL_SIZE_M ** 2           # 100 m² per pixel
PIXEL_AREA_KM2 = PIXEL_AREA_M2 / 1_000_000  # 0.0001 km² per pixel

print(f"Calculating flood area across {len(test_imgs)} test chips...")
print(f"Pixel size: {PIXEL_SIZE_M:.0f}m × {PIXEL_SIZE_M:.0f}m = {PIXEL_AREA_M2:.0f} m² per pixel")
print()

for img_path in test_imgs:
    with rasterio.open(img_path) as src:
        image = src.read()
        image = np.nan_to_num(image, nan=0.0)
        image = np.clip(image, -30, 0)
        image = (image - (-15.0)) / 10.0

    input_tensor = torch.from_numpy(image).float().unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        output = torch.sigmoid(model(input_tensor))
        mask = (output > 0.5).cpu().numpy().astype(np.uint8)
        flood_pixels = np.count_nonzero(mask)
        total_flood_pixels += flood_pixels
        total_area_km2     += flood_pixels * PIXEL_AREA_KM2

print("=" * 40)
print(f"📊 TOTAL TEST SET FLOOD SUMMARY")
print(f"   Test chips:          {len(test_imgs)}")
print(f"   Total flood pixels:  {total_flood_pixels:,}")
print(f"   Total flooded area:  {total_area_km2:.2f} km²")
print(f"   Average per chip:    {total_area_km2/len(test_imgs):.2f} km²")
print("=" * 40)
